# Data extraction

In [18]:
#!/usr/bin/env python3
"""
Extract all GNN datasets into pandas DataFrames for training.
Each dataset produces multiple DataFrames (nodes, edges, tasks, platforms, etc.)
Stores all in a dictionary structure for iterative processing.
"""

import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any
import warnings
warnings.filterwarnings('ignore')


def extract_dataset_to_dataframes(optimal_result_path: Path) -> Dict[str, pd.DataFrame]:
    """
    Extract a single optimal_result.json file into multiple DataFrames.
    
    Returns:
        Dictionary containing all DataFrames for this dataset
    """
    with open(optimal_result_path, "r") as f:
        result = json.load(f)
    
    # Get dataset ID from path
    dataset_id = optimal_result_path.parent.name
    
    # Get infrastructure and stats
    infra_nodes = result.get("config", {}).get("infrastructure", {}).get("nodes", [])
    stats = result.get("stats", {})
    task_results = stats.get("taskResults", [])
    placement_plan = result.get("sample", {}).get("placement_plan", {})
    
    # ========================================================================
    # 1. NODES DataFrame
    # ========================================================================
    nodes_data = []
    for i, node in enumerate(infra_nodes):
        node_name = node.get("node_name", f"node_{i}")
        node_type = node.get("type", "unknown")
        platforms = node.get("platforms", [])
        network_map = node.get("network_map", {})
        
        platform_counts = {}
        for p in platforms:
            platform_counts[p] = platform_counts.get(p, 0) + 1
        
        nodes_data.append({
            'node_id': i,
            'node_name': node_name,
            'node_type': node_type,
            'is_client': node_name.startswith('client_node'),
            'memory_gb': node.get("memory", 0),
            'n_platforms': len(platforms),
            'n_storage': len(node.get("storage", [])),
            'n_connections': len(network_map),
            'platform_types': ','.join(sorted(set(platforms))),
            'n_rpiCpu': platform_counts.get('rpiCpu', 0),
            'n_xavierCpu': platform_counts.get('xavierCpu', 0),
            'n_xavierGpu': platform_counts.get('xavierGpu', 0),
            'n_xavierDla': platform_counts.get('xavierDla', 0),
            'n_pynqFpga': platform_counts.get('pynqFpga', 0)
        })
    
    df_nodes = pd.DataFrame(nodes_data)
    
    # ========================================================================
    # 2. EDGES DataFrame
    # ========================================================================
    edges_data = []
    network_topology = stats.get("networkTopology", {})
    
    for src_node, connections in network_topology.items():
        if isinstance(connections, dict):
            for dst_node, latency in connections.items():
                edges_data.append({
                    'src_node': src_node,
                    'dst_node': dst_node,
                    'latency': latency,
                    'src_is_client': src_node.startswith('client_node'),
                    'dst_is_client': dst_node.startswith('client_node')
                })
    
    df_edges = pd.DataFrame(edges_data)
    
    # ========================================================================
    # 3. TASKS DataFrame
    # ========================================================================
    tasks_data = []
    for task_result in task_results:
        task_id = task_result.get("taskId")
        if task_id < 0:
            continue
        placement = placement_plan.get(str(task_id), [None, None])
        
        if isinstance(placement, list) and len(placement) >= 2:
            opt_node_id, opt_platform_id = placement[0], placement[1]
        else:
            opt_node_id, opt_platform_id = None, None
        
        tasks_data.append({
            'task_id': task_id,
            'task_type': task_result.get("taskType", {}).get("name", "unknown"),
            'source_node': task_result.get("sourceNode", ""),
            'dispatched_time': task_result.get("dispatchedTime", 0),
            'scheduled_time': task_result.get("scheduledTime", 0),
            'arrived_time': task_result.get("arrivedTime", 0),
            'started_time': task_result.get("startedTime", 0),
            'done_time': task_result.get("doneTime", 0),
            'optimal_node_id': opt_node_id,
            'optimal_platform_id': opt_platform_id,
            'optimal_platform_type': task_result.get("platform", {}).get("shortName", ""),
            'elapsed_time': task_result.get("elapsedTime", 0),
            'cold_start_time': task_result.get("coldStartTime", 0),
            'execution_time': task_result.get("executionTime", 0),
            'queue_time': task_result.get("queueTime", 0),
            'wait_time': task_result.get("waitTime", 0),
            'network_latency': task_result.get("networkLatency", 0),
            'compute_time': task_result.get("computeTime", 0),
            'communications_time': task_result.get("communicationsTime", 0),
            'cold_started': task_result.get("coldStarted", False),
            'cache_hit': task_result.get("cacheHit", False),
            'local_dependencies': task_result.get("localDependencies", False),
            'penalty': task_result.get("penalty", False),
            'energy': task_result.get("energy", 0)
        })
    
    df_tasks = pd.DataFrame(tasks_data)
    
    # ========================================================================
    # 4. PLATFORMS DataFrame
    # ========================================================================
    platforms_data = []
    node_results = stats.get("nodeResults", [])
    
    # Get replica state
    system_state = stats.get("systemStateResults", [{}])[-1] if stats.get("systemStateResults") else {}
    replicas_by_task = system_state.get("replicas", {})
    
    for node_result in node_results:
        node_id = node_result.get("nodeId")
        node_name = infra_nodes[node_id].get("node_name") if node_id < len(infra_nodes) else f"node_{node_id}"
        
        for plat_result in node_result.get("platformResults", []):
            plat_id = plat_result.get("platformId")
            plat_type_info = plat_result.get("platformType", {})
            plat_type = plat_type_info.get("shortName", "unknown")
            
            # Check replica state
            has_dnn1_replica = False
            has_dnn2_replica = False
            
            for task_type, replica_list in replicas_by_task.items():
                if isinstance(replica_list, list):
                    for replica in replica_list:
                        if isinstance(replica, list) and len(replica) >= 2:
                            if replica[0] == node_name and replica[1] == plat_id:
                                if task_type == "dnn1":
                                    has_dnn1_replica = True
                                elif task_type == "dnn2":
                                    has_dnn2_replica = True
            
            platforms_data.append({
                'platform_id': plat_id,
                'node_id': node_id,
                'node_name': node_name,
                'platform_type': plat_type,
                'hardware': plat_type_info.get("hardware", ""),
                'price': plat_type_info.get("price", 0),
                'idle_energy': plat_type_info.get("idleEnergy", 0),
                'has_dnn1_replica': has_dnn1_replica,
                'has_dnn2_replica': has_dnn2_replica,
                'energy_consumed': plat_result.get("energy", 0),
                'idle_time': plat_result.get("idleTime", 0),
                'idle_proportion': plat_result.get("idleProportion", 0),
                'storage_time': plat_result.get("storageTime", 0)
            })
    
    df_platforms = pd.DataFrame(platforms_data)
    
    # ========================================================================
    # 5. TASK_COMPATIBILITY DataFrame
    # ========================================================================
    task_types = result.get("sim_inputs", {}).get("task_types", {})
    compatibility_data = []
    
    for task_name, task_config in task_types.items():
        supported_platforms = task_config.get("platforms", [])
        memory_reqs = task_config.get("memoryRequirements", {})
        exec_times = task_config.get("executionTime", {})
        cold_start_times = task_config.get("coldStartDuration", {})
        image_sizes = task_config.get("imageSize", {})
        
        for plat_type in supported_platforms:
            compatibility_data.append({
                'task_type': task_name,
                'platform_type': plat_type,
                'memory_required': memory_reqs.get(plat_type, 0),
                'execution_time': exec_times.get(plat_type, 0),
                'cold_start_duration': cold_start_times.get(plat_type, 0),
                'image_size': image_sizes.get(plat_type, 0)
            })
    
    df_compatibility = pd.DataFrame(compatibility_data)
    
    # ========================================================================
    # 6. PLACEMENTS DataFrame (Ground Truth Labels)
    # ========================================================================
    placements_data = []
    for task_id_str, placement in placement_plan.items():
        if isinstance(placement, list) and len(placement) >= 2:
            node_id, platform_id = placement[0], placement[1]
            task_result = next((tr for tr in task_results if tr.get("taskId") == int(task_id_str)), None)
            
            placements_data.append({
                'task_id': int(task_id_str),
                'optimal_node_id': node_id,
                'optimal_platform_id': platform_id,
                'node_name': infra_nodes[node_id].get("node_name") if node_id < len(infra_nodes) else "",
                'platform_type': task_result.get("platform", {}).get("shortName", "") if task_result else "",
                'task_type': task_result.get("taskType", {}).get("name", "") if task_result else "",
                'source_node': task_result.get("sourceNode", "") if task_result else "",
                'is_local': infra_nodes[node_id].get("node_name") == (task_result.get("sourceNode", "") if task_result else "") if node_id < len(infra_nodes) else False,
                'rtt': task_result.get("elapsedTime", 0) if task_result else 0
            })
    
    df_placements = pd.DataFrame(placements_data)
    
    # ========================================================================
    # 7. REPLICAS DataFrame
    # ========================================================================
    replicas_data = []
    for task_type, replica_list in replicas_by_task.items():
        if isinstance(replica_list, list):
            for replica in replica_list:
                if isinstance(replica, list) and len(replica) >= 2:
                    node_name, platform_id = replica[0], replica[1]
                    node_id = next((i for i, n in enumerate(infra_nodes) if n.get("node_name") == node_name), None)
                    
                    replicas_data.append({
                        'task_type': task_type,
                        'node_id': node_id,
                        'node_name': node_name,
                        'platform_id': platform_id,
                        'is_client': node_name.startswith('client_node')
                    })
    
    df_replicas = pd.DataFrame(replicas_data)
    
    # ========================================================================
    # 8. CONFIGURATION DataFrame
    # ========================================================================
    infra_config = result.get("sample", {}).get("infra_config", {})
    config_data = {
        'dataset_id': dataset_id,
        'connection_probability': infra_config.get("network", {}).get("topology", {}).get("connection_probability", 0),
        'replicas_per_client_dnn1': infra_config.get("replicas", {}).get("dnn1", {}).get("per_client", 0),
        'replicas_per_server_dnn1': infra_config.get("replicas", {}).get("dnn1", {}).get("per_server", 0),
        'replicas_per_client_dnn2': infra_config.get("replicas", {}).get("dnn2", {}).get("per_client", 0),
        'replicas_per_server_dnn2': infra_config.get("replicas", {}).get("dnn2", {}).get("per_server", 0),
        'preinit_client_pct': infra_config.get("preinit", {}).get("client_percentage", 0),
        'preinit_server_pct': infra_config.get("preinit", {}).get("server_percentage", 0),
        'prewarm_queue_dnn1': infra_config.get("prewarm", {}).get("dnn1", {}).get("initial_queue", 0),
        'prewarm_queue_dnn2': infra_config.get("prewarm", {}).get("dnn2", {}).get("initial_queue", 0),
        'network_bandwidth': result.get("config", {}).get("infrastructure", {}).get("network", {}).get("bandwidth", 0),
        'total_nodes': len(infra_nodes),
        'total_client_nodes': sum(1 for n in infra_nodes if n.get("node_name", "").startswith("client_node")),
        'total_server_nodes': sum(1 for n in infra_nodes if not n.get("node_name", "").startswith("client_node"))
    }
    
    df_config = pd.DataFrame([config_data])
    
    # ========================================================================
    # 9. SUMMARY_METRICS DataFrame
    # ========================================================================
    # Try to get RTT from best.json
    best_json_path = optimal_result_path.parent / "best.json"
    best_rtt = None
    if best_json_path.exists():
        try:
            with open(best_json_path, "r") as f:
                best_rtt = json.load(f).get("rtt")
        except:
            pass
    
    # Fallback to sum of task elapsed times
    if best_rtt is None and task_results:
        best_rtt = sum(tr.get("elapsedTime", 0) for tr in task_results)
    
    metrics_data = {
        'dataset_id': dataset_id,
        'total_rtt': best_rtt or 0,
        'avg_elapsed_time': stats.get("averageElapsedTime", 0),
        'avg_cold_start_time': stats.get("averageColdStartTime", 0),
        'avg_execution_time': stats.get("averageExecutionTime", 0),
        'avg_wait_time': stats.get("averageWaitTime", 0),
        'avg_queue_time': stats.get("averageQueueTime", 0),
        'avg_compute_time': stats.get("averageComputeTime", 0),
        'avg_communications_time': stats.get("averageCommunicationsTime", 0),
        'avg_network_latency': stats.get("averageNetworkLatency", 0),
        'total_energy': stats.get("energy", 0),
        'reclaimable_energy': stats.get("reclaimableEnergy", 0),
        'penalty_proportion': stats.get("penaltyProportion", 0),
        'cold_start_proportion': stats.get("coldStartProportion", 0),
        'cache_hit_proportion': stats.get("taskCacheHitsProportion", 0),
        'offloading_rate': stats.get("offloadingRate", 0),
        'unused_platforms_pct': stats.get("unusedPlatforms", 0),
        'unused_nodes_pct': stats.get("unusedNodes", 0),
        'end_time': stats.get("endTime", 0),
        'n_tasks': len(task_results)
    }
    
    df_metrics = pd.DataFrame([metrics_data])
    
    # Return all DataFrames for this dataset
    return {
        'nodes': df_nodes,
        'edges': df_edges,
        'tasks': df_tasks,
        'platforms': df_platforms,
        'placements': df_placements,
        'compatibility': df_compatibility,
        'replicas': df_replicas,
        'config': df_config,
        'metrics': df_metrics
    }


def extract_all_datasets(base_dir: Path, max_datasets: int = None) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    Extract all datasets from gnn_datasets directory.
    
    Args:
        base_dir: Path to gnn_datasets directory
        max_datasets: Optional limit on number of datasets to process
    
    Returns:
        Dictionary mapping dataset_id to its DataFrames dictionary
    """
    all_datasets = {}
    dataset_dirs = sorted(base_dir.glob("ds_*"))
    
    if max_datasets:
        dataset_dirs = dataset_dirs[:max_datasets]
    
    print(f"Found {len(dataset_dirs)} dataset directories")
    print(f"Processing datasets...\n")
    
    successful = 0
    skipped = 0
    
    for i, dataset_dir in enumerate(dataset_dirs):
        dataset_id = dataset_dir.name
        optimal_result_path = dataset_dir / "optimal_result.json"
        
        if not optimal_result_path.exists():
            skipped += 1
            if i < 10 or i % 100 == 0:
                print(f"  [{i+1:4d}/{len(dataset_dirs)}] {dataset_id}: SKIP (no optimal_result.json)")
            continue
        
        try:
            dataframes = extract_dataset_to_dataframes(optimal_result_path)
            all_datasets[dataset_id] = dataframes
            successful += 1
            
            if i < 10 or i % 100 == 0:
                rtt = dataframes['metrics']['total_rtt'].values[0]
                n_tasks = dataframes['metrics']['n_tasks'].values[0]
                print(f"  [{i+1:4d}/{len(dataset_dirs)}] {dataset_id}: OK (RTT={rtt:.3f}s, tasks={int(n_tasks)})")
        
        except Exception as e:
            skipped += 1
            if i < 10:
                print(f"  [{i+1:4d}/{len(dataset_dirs)}] {dataset_id}: ERROR - {str(e)}")
    
    print(f"\n{'='*80}")
    print(f"Extraction Complete!")
    print(f"{'='*80}")
    print(f"  Successful: {successful}")
    print(f"  Skipped:    {skipped}")
    print(f"  Total:      {len(dataset_dirs)}")
    
    return all_datasets


def create_consolidated_dataframes(all_datasets: Dict[str, Dict[str, pd.DataFrame]]) -> Dict[str, pd.DataFrame]:
    """
    Consolidate all datasets into single DataFrames with dataset_id as key.
    
    Returns:
        Dictionary of consolidated DataFrames (one per table type)
    """
    print(f"\n{'='*80}")
    print("Creating Consolidated DataFrames...")
    print(f"{'='*80}\n")
    
    consolidated = {}
    
    # Tables that should be concatenated across datasets
    tables_to_consolidate = ['config', 'metrics', 'placements', 'tasks']
    
    for table_name in tables_to_consolidate:
        dfs_to_concat = []
        
        for dataset_id, dataframes in all_datasets.items():
            if table_name in dataframes:
                df = dataframes[table_name].copy()
                # Add dataset_id if not present
                if 'dataset_id' not in df.columns:
                    df.insert(0, 'dataset_id', dataset_id)
                dfs_to_concat.append(df)
        
        if dfs_to_concat:
            consolidated[table_name] = pd.concat(dfs_to_concat, ignore_index=True)
            print(f"  {table_name:<20} : {consolidated[table_name].shape}")
    
    return consolidated


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Configuration
    BASE_DIR = Path("/root/projects/my-herosim/simulation_data/gnn_datasets")
    
    # Extract all datasets
    print("="*80)
    print("EXTRACTING ALL GNN DATASETS")
    print("="*80)
    print()
    
    all_datasets = extract_all_datasets(BASE_DIR, max_datasets=None)
    
    # Create consolidated DataFrames
    consolidated = create_consolidated_dataframes(all_datasets)
    
    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    print(f"\n{'='*80}")
    print("FINAL DATASET STRUCTURE")
    print(f"{'='*80}\n")
    
    print("1. PER-DATASET DataFrames (all_datasets dict):")
    print(f"   - Keys: {len(all_datasets)} dataset IDs (ds_0000, ds_0001, ...)")
    print("   - Each dataset contains:")
    print("     * nodes (20 rows) - Infrastructure nodes")
    print("     * edges (varies) - Network connectivity")
    print("     * tasks (5 rows) - Workload tasks with labels")
    print("     * platforms (98 rows) - Platform state")
    print("     * placements (5 rows) - Optimal placement labels")
    print("     * compatibility (6 rows) - Task-platform constraints")
    print("     * replicas (varies) - Warm replica distribution")
    print("     * config (1 row) - Configuration parameters")
    print("     * metrics (1 row) - Performance summary")
    print()
    
    print("2. CONSOLIDATED DataFrames (consolidated dict):")
    for name, df in consolidated.items():
        print(f"   - {name:<20} : {df.shape}")
    print()
    
    print("="*80)
    print("USAGE EXAMPLES")
    print("="*80)
    print("""
# Access a specific dataset:
ds_0049_data = all_datasets['ds_0049']
df_nodes = ds_0049_data['nodes']
df_tasks = ds_0049_data['tasks']

# Access consolidated data across all datasets:
all_configs = consolidated['config']
all_metrics = consolidated['metrics']
all_placements = consolidated['placements']

# Iterate through all datasets:
for dataset_id, dataframes in all_datasets.items():
    print(f"{dataset_id}: RTT = {dataframes['metrics']['total_rtt'].values[0]:.3f}s")

# Find best performing dataset:
best_dataset_id = consolidated['metrics'].loc[consolidated['metrics']['total_rtt'].idxmin(), 'dataset_id']
print(f"Best dataset: {best_dataset_id}")

# Analyze configuration impact:
config_rtt = consolidated['config'].merge(consolidated['metrics'], on='dataset_id')
print(config_rtt[['connection_probability', 'prewarm_queue_dnn1', 'total_rtt']].corr())
""")
    
    print("\n" + "="*80)
    print("DATA READY FOR GNN TRAINING!")
    print("="*80)
    print(f"  Total datasets: {len(all_datasets)}")
    print(f"  Total tasks (training samples): {len(consolidated.get('tasks', pd.DataFrame()))}")
    print(f"  Total placements (labels): {len(consolidated.get('placements', pd.DataFrame()))}")
    print()
    
    # Save summary statistics
    if consolidated.get('metrics') is not None:
        print("Dataset Performance Summary:")
        print(f"  RTT range: {consolidated['metrics']['total_rtt'].min():.3f}s - {consolidated['metrics']['total_rtt'].max():.3f}s")
        print(f"  RTT mean:  {consolidated['metrics']['total_rtt'].mean():.3f}s")
        print(f"  RTT std:   {consolidated['metrics']['total_rtt'].std():.3f}s")

EXTRACTING ALL GNN DATASETS

Found 122 dataset directories
Processing datasets...

  [   1/122] ds_0000: SKIP (no optimal_result.json)
  [   2/122] ds_0001: SKIP (no optimal_result.json)
  [   3/122] ds_0002: SKIP (no optimal_result.json)
  [   4/122] ds_0003: SKIP (no optimal_result.json)
  [   5/122] ds_0004: SKIP (no optimal_result.json)
  [   6/122] ds_0005: SKIP (no optimal_result.json)
  [   7/122] ds_0006: SKIP (no optimal_result.json)
  [   8/122] ds_0007: SKIP (no optimal_result.json)
  [   9/122] ds_0008: SKIP (no optimal_result.json)
  [  10/122] ds_0009: SKIP (no optimal_result.json)
  [ 101/122] ds_0100: SKIP (no optimal_result.json)

Extraction Complete!
  Successful: 50
  Skipped:    72
  Total:      122

Creating Consolidated DataFrames...

  config               : (50, 14)
  metrics              : (50, 20)
  placements           : (250, 10)
  tasks                : (250, 25)

FINAL DATASET STRUCTURE

1. PER-DATASET DataFrames (all_datasets dict):
   - Keys: 50 dataset 

### load

In [19]:
def load_all_datasets(base_dir: Path) -> Dict[str, Dict[str, pd.DataFrame]]:
    """Load all datasets from gnn_datasets directory."""
    all_datasets = {}
    dataset_dirs = sorted(base_dir.glob("ds_*"))
    
    print(f"Loading {len(dataset_dirs)} datasets...")
    
    for dataset_dir in dataset_dirs:
        optimal_result_path = dataset_dir / "optimal_result.json"
        if not optimal_result_path.exists():
            continue
        
        try:
            dataframes = extract_dataset_to_dataframes(optimal_result_path)
            all_datasets[dataset_dir.name] = dataframes
        except Exception as e:
            print(f"  Error loading {dataset_dir.name}: {e}")
    
    print(f"Loaded {len(all_datasets)} datasets successfully\n")
    return all_datasets


In [20]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GIN
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

### build graphs

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION
# ============================================================================

def build_graph(df_nodes, df_tasks, df_platforms) -> Data:
    """
    Build a bipartite graph with tasks and platforms as nodes.
    Edges connect tasks to feasible platforms based on network connectivity.
    
    TODO: Add task-platform compatibility filtering (e.g., dnn1 can only run on certain platform types)
    """
    
    # Create node ID mappings
    # Task nodes: 0 to n_tasks-1
    # Platform nodes: n_tasks to n_tasks+n_platforms-1
    n_tasks = len(df_tasks)
    n_platforms = len(df_platforms)
    
    task_offset = 0
    platform_offset = n_tasks
    
    # ========================================================================
    # TASK FEATURES: [task_type_onehot, source_node_id]
    # ========================================================================
    task_types = ['dnn1', 'dnn2']
    task_features = []
    
    for _, task in df_tasks.iterrows():
        # One-hot encode task type
        task_type_onehot = [1.0 if task['task_type'] == tt else 0.0 for tt in task_types]
        
        # Source node ID (normalized to [0, 1])
        source_node_id = df_nodes[df_nodes['node_name'] == task['source_node']].index[0] if len(df_nodes[df_nodes['node_name'] == task['source_node']]) > 0 else 0
        source_node_id_norm = source_node_id / len(df_nodes)
        
        task_features.append(task_type_onehot + [source_node_id_norm])
    
    task_features = torch.tensor(task_features, dtype=torch.float)
    
    # ========================================================================
    # PLATFORM FEATURES: [platform_type_onehot, has_dnn1_replica, has_dnn2_replica]
    # ========================================================================
    platform_types = ['rpiCpu', 'xavierCpu', 'xavierGpu', 'xavierDla', 'pynqFpga']
    platform_features = []
    
    for _, platform in df_platforms.iterrows():
        # One-hot encode platform type
        plat_type_onehot = [1.0 if platform['platform_type'] == pt else 0.0 for pt in platform_types]
        
        # Replica flags
        has_dnn1 = 1.0 if platform['has_dnn1_replica'] else 0.0
        has_dnn2 = 1.0 if platform['has_dnn2_replica'] else 0.0
        
        platform_features.append(plat_type_onehot + [has_dnn1, has_dnn2])
    
    platform_features = torch.tensor(platform_features, dtype=torch.float)
    
    # Store raw features separately (they have different dimensions)
    # Tasks: dim=3, Platforms: dim=7
    # We'll encode them separately in the model, so don't concatenate here
    task_features_tensor = task_features
    platform_features_tensor = platform_features
    
    # ========================================================================
    # EDGES: Task → Feasible Platforms (based on network connectivity)
    # ========================================================================
    edge_index = []
    edge_labels = []  # For tracking which edges are optimal
    
    for task_idx, task in df_tasks.iterrows():
        source_node_name = task['source_node']
        task_type = task['task_type']
        
        # Get source node's network map
        source_node_row = df_nodes[df_nodes['node_name'] == source_node_name]
        if len(source_node_row) == 0:
            continue
        
        network_map = source_node_row.iloc[0]['network_map']
        
        # Feasible nodes: source node + connected server nodes
        feasible_node_names = [source_node_name] + list(network_map.keys())
        
        # Find all platforms on feasible nodes
        # TODO: Add compatibility filtering here (task_type → supported platform types)
        feasible_platforms = df_platforms[df_platforms['node_name'].isin(feasible_node_names)]
        
        # Create edges from task to each feasible platform
        task_node_idx = task_offset + task_idx
        
        for _, platform in feasible_platforms.iterrows():
            platform_idx = platform_offset + platform.name  # platform.name is the DataFrame index
            
            edge_index.append([task_node_idx, platform_idx])
            
            # Mark if this is the optimal edge (for ground truth)
            is_optimal = (platform['platform_id'] == task['optimal_platform_id'])
            edge_labels.append(is_optimal)
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # ========================================================================
    # LABELS: For each task, which platform is optimal?
    # ========================================================================
    # We need to convert optimal_platform_id to the index in our edge list
    y = []
    
    for task_idx, task in df_tasks.iterrows():
        task_node_idx = task_offset + task_idx
        optimal_platform_id = task['optimal_platform_id']
        
        # Find which edge index corresponds to this task's optimal platform
        task_edges_mask = (edge_index[0] == task_node_idx)
        task_edge_indices = torch.where(task_edges_mask)[0]
        
        # Find the optimal edge among this task's edges
        optimal_edge_idx = None
        for edge_idx in task_edge_indices:
            platform_node_idx = edge_index[1, edge_idx].item()
            platform_df_idx = platform_node_idx - platform_offset
            
            if platform_df_idx >= 0 and platform_df_idx < len(df_platforms):
                platform_id = df_platforms.iloc[platform_df_idx]['platform_id']
                if platform_id == optimal_platform_id:
                    # Label is the index within this task's feasible platforms
                    optimal_edge_idx = (task_edge_indices == edge_idx).nonzero(as_tuple=True)[0].item()
                    break
        
        y.append(optimal_edge_idx if optimal_edge_idx is not None else 0)
    
    y = torch.tensor(y, dtype=torch.long)
    
    # ========================================================================
    # CREATE PyG Data object
    # ========================================================================
    data = Data(
        edge_index=edge_index,
        y=y,
        n_tasks=n_tasks,
        n_platforms=n_platforms,
        # Store task and platform features separately (different dimensions)
        task_features=task_features_tensor,
        platform_features=platform_features_tensor
    )
    
    return data

In [38]:
# ============================================================================
# GNN MODEL
# ============================================================================

class TaskEncoder(nn.Module):
    """2-layer MLP encoder for task features."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class PlatformEncoder(nn.Module):
    """2-layer MLP encoder for platform features."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class EdgeScorer(nn.Module):
    """2-layer MLP to score task-platform edges."""
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        # Input: concatenation of task and platform embeddings
        self.fc1 = nn.Linear(2 * embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    
    def forward(self, e_task, e_platform):
        # Concatenate task and platform embeddings
        x = torch.cat([e_task, e_platform], dim=-1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x.squeeze(-1)


class TaskPlacementGNN(nn.Module):
    """
    GNN for task-to-platform placement prediction.
    
    Architecture:
    1. Encode task and platform features separately
    2. GIN to produce node embeddings
    3. Edge MLP to score task-platform compatibility
    4. Masked softmax to predict placement probabilities
    """
    def __init__(self, task_feature_dim, platform_feature_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()
        
        self.embedding_dim = embedding_dim
        
        # Separate encoders for tasks and platforms
        self.task_encoder = TaskEncoder(task_feature_dim, hidden_dim, embedding_dim)
        self.platform_encoder = PlatformEncoder(platform_feature_dim, hidden_dim, embedding_dim)
        
        # GIN model for message passing
        self.gin = GIN(
            in_channels=embedding_dim,
            hidden_channels=hidden_dim,
            num_layers=3,
            out_channels=embedding_dim
        )
        
        # Edge scoring MLP
        self.edge_scorer = EdgeScorer(embedding_dim, hidden_dim)
    
    def forward(self, data):
        n_tasks = data.n_tasks
        n_platforms = data.n_platforms
        
        # Encode task and platform features separately (they have different input dims)
        task_embeddings = self.task_encoder(data.task_features)
        platform_embeddings = self.platform_encoder(data.platform_features)
        
        # Combine for GIN (now both have same embedding_dim)
        x = torch.cat([task_embeddings, platform_embeddings], dim=0)
        
        # Apply GIN for message passing
        x = self.gin(x, data.edge_index)
        
        # Split back into task and platform embeddings
        task_emb = x[:n_tasks]
        platform_emb = x[n_tasks:]
        
        # Score all edges
        edge_scores = []
        for i in range(data.edge_index.size(1)):
            task_idx = data.edge_index[0, i].item()
            platform_idx = data.edge_index[1, i].item() - n_tasks
            
            if platform_idx >= 0 and platform_idx < n_platforms:
                score = self.edge_scorer(task_emb[task_idx], platform_emb[platform_idx])
                edge_scores.append(score)
        
        edge_scores = torch.stack(edge_scores)
        
        # Compute masked softmax per task
        # For each task, normalize over its feasible platforms
        logits_per_task = []
        
        for task_idx in range(n_tasks):
            task_edges_mask = (data.edge_index[0] == task_idx)
            task_edge_scores = edge_scores[task_edges_mask]
            
            # Softmax over this task's feasible platforms
            task_probs = F.softmax(task_edge_scores, dim=0)
            logits_per_task.append(task_edge_scores)
        
        return logits_per_task

In [39]:
# ============================================================================
# TRAINING LOOP
# ============================================================================

def train_epoch(model, train_graphs, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for data in train_graphs:
        data = data.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        logits_per_task = model(data)
        
        # Compute cross-entropy loss for each task
        loss = 0
        for task_idx, task_logits in enumerate(logits_per_task):
            # Ground truth: which platform (among feasible ones) is optimal
            target = data.y[task_idx]
            
            if target >= 0 and target < len(task_logits):
                # Cross-entropy loss (negative log probability of correct class)
                task_loss = F.cross_entropy(task_logits.unsqueeze(0), target.unsqueeze(0))
                loss += task_loss
        
        # Average loss over tasks
        loss = loss / len(logits_per_task)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_graphs)


def evaluate(model, graphs, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data in graphs:
            data = data.to(device)
            
            # Forward pass
            logits_per_task = model(data)
            
            # Compute loss and accuracy
            for task_idx, task_logits in enumerate(logits_per_task):
                target = data.y[task_idx]
                
                if target >= 0 and target < len(task_logits):
                    task_loss = F.cross_entropy(task_logits.unsqueeze(0), target.unsqueeze(0))
                    total_loss += task_loss.item()
                    
                    # Accuracy: did we predict the correct platform?
                    pred = task_logits.argmax().item()
                    correct += (pred == target.item())
                    total += 1
    
    avg_loss = total_loss / total if total > 0 else 0
    accuracy = correct / total if total > 0 else 0
    
    return avg_loss, accuracy

In [40]:
# ============================================================================
# MAIN TRAINING SCRIPT
# ============================================================================

if __name__ == "__main__":
    print("="*80)
    print("GNN TASK PLACEMENT TRAINING")
    print("="*80)
    print()
    
    # Configuration
    BASE_DIR = Path("/root/projects/my-herosim/simulation_data/gnn_datasets")
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Hyperparameters
    EMBEDDING_DIM = 64
    HIDDEN_DIM = 128
    LEARNING_RATE = 0.001
    EPOCHS = 100
    
    print(f"Device: {DEVICE}")
    print(f"Embedding dim: {EMBEDDING_DIM}")
    print(f"Hidden dim: {HIDDEN_DIM}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Epochs: {EPOCHS}")
    print()
    
    # ========================================================================
    # Load all datasets
    # ========================================================================
    all_datasets = load_all_datasets(BASE_DIR)
    
    if len(all_datasets) == 0:
        print("ERROR: No datasets loaded!")
        exit(1)
    
    # ========================================================================
    # Build graphs for all datasets
    # ========================================================================
    print("Building graphs...")
    graphs = []
    dataset_ids = []
    
    for dataset_id, dataframes in all_datasets.items():
        try:
            graph = build_graph(
                dataframes['nodes'],
                dataframes['tasks'],
                dataframes['platforms']
            )
            graphs.append(graph)
            dataset_ids.append(dataset_id)
        except Exception as e:
            print(f"  Error building graph for {dataset_id}: {e}")
    
    print(f"Built {len(graphs)} graphs\n")
    
    # ========================================================================
    # Train/Val/Test Split (80/20)
    # ========================================================================
    train_graphs, test_graphs, train_ids, test_ids = train_test_split(
        graphs, dataset_ids, test_size=0.2, random_state=42
    )
    
    print(f"Dataset split:")
    print(f"  Train: {len(train_graphs)} datasets")
    print(f"  Test:  {len(test_graphs)} datasets")
    print()
    
    # ========================================================================
    # Initialize model
    # ========================================================================
    # Task features: 2 (task types) + 1 (source node ID) = 3
    # Platform features: 5 (platform types) + 2 (replica flags) = 7
    task_feature_dim = 3
    platform_feature_dim = 7
    
    model = TaskPlacementGNN(
        task_feature_dim=task_feature_dim,
        platform_feature_dim=platform_feature_dim,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print()
    
    # ========================================================================
    # Training loop
    # ========================================================================
    print("="*80)
    print("TRAINING")
    print("="*80)
    print()
    
    best_val_acc = 0
    
    for epoch in range(EPOCHS):
        # Train
        train_loss = train_epoch(model, train_graphs, optimizer, DEVICE)
        
        # Evaluate
        val_loss, val_acc = evaluate(model, test_graphs, DEVICE)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            # Save best model
            torch.save(model.state_dict(), 'best_gnn_placement_model.pt')
        
        # Print progress
        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    print()
    print(f"Best validation accuracy: {best_val_acc*100:.2f}%")
    
    # ========================================================================
    # Final Evaluation
    # ========================================================================
    print()
    print("="*80)
    print("FINAL EVALUATION")
    print("="*80)
    
    # Load best model
    model.load_state_dict(torch.load('best_gnn_placement_model.pt'))
    
    train_loss, train_acc = evaluate(model, train_graphs, DEVICE)
    test_loss, test_acc = evaluate(model, test_graphs, DEVICE)
    
    print(f"\nTrain: Loss={train_loss:.4f}, Accuracy={train_acc*100:.2f}%")
    print(f"Test:  Loss={test_loss:.4f}, Accuracy={test_acc*100:.2f}%")
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)
    print(f"Model saved to: best_gnn_placement_model.pt")
    print(f"Best validation accuracy: {best_val_acc*100:.2f}%")

GNN TASK PLACEMENT TRAINING

Device: cpu
Embedding dim: 64
Hidden dim: 128
Learning rate: 0.001
Epochs: 100

Loading 122 datasets...
Loaded 50 datasets successfully

Building graphs...
  Error building graph for ds_0010: 0
  Error building graph for ds_0011: 0
  Error building graph for ds_0012: 0
  Error building graph for ds_0013: 0
  Error building graph for ds_0014: 0
  Error building graph for ds_0015: 0
  Error building graph for ds_0016: 0
  Error building graph for ds_0017: 0
  Error building graph for ds_0018: 0
  Error building graph for ds_0019: 0
  Error building graph for ds_0040: 0
  Error building graph for ds_0041: 0
  Error building graph for ds_0042: 0
  Error building graph for ds_0043: 0
  Error building graph for ds_0044: 0
  Error building graph for ds_0045: 0
  Error building graph for ds_0046: 0
  Error building graph for ds_0047: 0
  Error building graph for ds_0048: 0
  Error building graph for ds_0049: 0
  Error building graph for ds_0050: 0
  Error building 

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.